In [1]:
# If running outside a managed BQ notebook, authenticate:
# from google.colab import auth; auth.authenticate_user()

from google.cloud import bigquery

PROJECT_ID = None  # e.g. "my-gcp-project"; leave None to use the environment default
DATASET    = "fraud_detection"
LOCATION   = "US"

client = bigquery.Client(project=PROJECT_ID) if PROJECT_ID else bigquery.Client()
PROJECT_ID = client.project
print("Using project:", PROJECT_ID)

RAW_TABLE   = f"{PROJECT_ID}.{DATASET}.fraud_data_raw"
TRAIN_TABLE = f"{PROJECT_ID}.{DATASET}.fraud_training_data"

def run(sql):
    """Run a SQL statement and block until it finishes."""
    job = client.query(sql)
    job.result()
    return job


Using project: qwiklabs-gcp-01-9ed82c2a1f67


In [2]:
run(f"""
CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{DATASET}`
OPTIONS(location="{LOCATION}")
""")
print("Dataset ready:", DATASET)


Dataset ready: fraud_detection


In [3]:
run(f"""
LOAD DATA OVERWRITE `{RAW_TABLE}`
FROM FILES (
  format = 'CSV',
  uris   = ['gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv'],
  skip_leading_rows = 1,
  field_delimiter = ',',
  max_bad_records = 10
)
""")
print("Loaded raw table:", RAW_TABLE)


Loaded raw table: qwiklabs-gcp-01-9ed82c2a1f67.fraud_detection.fraud_data_raw


In [4]:
client.query(f"SELECT * FROM `{RAW_TABLE}` LIMIT 5").to_dataframe()

,Applicant_ID,Age,Employment_Status,Income,Number_of_Dependents,Amount_Requested,Previous_Assistance_Received,Previous_Assistance_Date,Supporting_Doc_Verified,Application_Frequency_Last_Year,IP_Address,Device_Type,Application_Date,Fraudulent
0,217,65,Unemployed,28984,4,5872,False,NaT,False,1,156.133.45.45,Mobile,2024-08-18,0
1,226,54,Self-Employed,0,1,6631,False,NaT,False,1,245.13.80.245,Tablet,2024-05-11,0
2,240,26,Self-Employed,64477,5,8612,False,NaT,True,1,213.103.170.95,Mobile,2024-08-14,0
3,252,28,Unemployed,28576,4,2951,False,NaT,True,1,234.179.149.207,Desktop,2024-06-12,0
4,266,43,Employed,44930,5,2324,False,NaT,False,1,66.109.96.227,Mobile,2024-08-16,0


In [5]:
cats = client.query(f"""
SELECT 'Employment_Status' AS field, Employment_Status AS value
FROM `{RAW_TABLE}` WHERE Employment_Status IS NOT NULL
GROUP BY value
UNION ALL
SELECT 'Device_Type', Device_Type
FROM `{RAW_TABLE}` WHERE Device_Type IS NOT NULL
GROUP BY Device_Type
ORDER BY field, value
""").to_dataframe()
cats


,field,value
0,Device_Type,Desktop
1,Device_Type,Mobile
2,Device_Type,Tablet
3,Employment_Status,Employed
4,Employment_Status,Self-Employed
5,Employment_Status,Unemployed


In [6]:
import re

def sanitize(v):
    return re.sub(r"[^0-9A-Za-z]+", "_", str(v)).strip("_")

emp_vals = cats.loc[cats.field == "Employment_Status", "value"].tolist()
dev_vals = cats.loc[cats.field == "Device_Type", "value"].tolist()

def one_hot(field, values):
    lines = []
    for v in values:
        col = f"{field}_{sanitize(v)}"
        lines.append(
            f"    CASE WHEN {field} = '{v}' THEN 1 ELSE 0 END AS {col}"
        )
    return ",\n".join(lines)

emp_oh = one_hot("Employment_Status", emp_vals)
dev_oh = one_hot("Device_Type", dev_vals)
print(emp_oh, "\n")
print(dev_oh)


    CASE WHEN Employment_Status = 'Employed' THEN 1 ELSE 0 END AS Employment_Status_Employed,
    CASE WHEN Employment_Status = 'Self-Employed' THEN 1 ELSE 0 END AS Employment_Status_Self_Employed,
    CASE WHEN Employment_Status = 'Unemployed' THEN 1 ELSE 0 END AS Employment_Status_Unemployed 

    CASE WHEN Device_Type = 'Desktop' THEN 1 ELSE 0 END AS Device_Type_Desktop,
    CASE WHEN Device_Type = 'Mobile' THEN 1 ELSE 0 END AS Device_Type_Mobile,
    CASE WHEN Device_Type = 'Tablet' THEN 1 ELSE 0 END AS Device_Type_Tablet


In [7]:
age_bins = """    CASE WHEN Age BETWEEN 18 AND 24 THEN 1 ELSE 0 END AS Age_18_24,
    CASE WHEN Age BETWEEN 25 AND 34 THEN 1 ELSE 0 END AS Age_25_34,
    CASE WHEN Age BETWEEN 35 AND 44 THEN 1 ELSE 0 END AS Age_35_44,
    CASE WHEN Age BETWEEN 45 AND 54 THEN 1 ELSE 0 END AS Age_45_54,
    CASE WHEN Age BETWEEN 55 AND 64 THEN 1 ELSE 0 END AS Age_55_64,
    CASE WHEN Age >= 65 THEN 1 ELSE 0 END AS Age_65_plus"""

feature_sql = f"""
CREATE OR REPLACE TABLE `{{TRAIN_TABLE}}` AS
SELECT
    Applicant_ID,
    Age,
    Income,
    Number_of_Dependents,
    Amount_Requested,
    Application_Frequency_Last_Year,
    Fraudulent,

    -- One-hot: Employment_Status
{{emp_oh}},

    -- One-hot: Device_Type
{{dev_oh}},

    -- Age bins (one-hot)
{{age_bins}},

    -- Income to amount-requested ratio (guard divide-by-zero)
    SAFE_DIVIDE(Income, Amount_Requested) AS Income_to_Amount_Requested,

    -- Days between previous assistance and this application
    DATE_DIFF(Application_Date, Previous_Assistance_Date, DAY) AS Time_Since_Previous_Assistance_Raw,
    IFNULL(DATE_DIFF(Application_Date, Previous_Assistance_Date, DAY), 0) AS Time_Since_Previous_Assistance,
    CASE WHEN Previous_Assistance_Date IS NOT NULL THEN 1 ELSE 0 END AS Has_Previous_Assistance,

    -- True/False -> 0/1
    CASE WHEN Previous_Assistance_Received THEN 1 ELSE 0 END AS Previous_Assistance_Received,
    CASE WHEN Supporting_Doc_Verified     THEN 1 ELSE 0 END AS Supporting_Doc_Verified

FROM `{{RAW_TABLE}}`
""".format(TRAIN_TABLE=TRAIN_TABLE, emp_oh=emp_oh, dev_oh=dev_oh,
           age_bins=age_bins, RAW_TABLE=RAW_TABLE)

print(feature_sql)



CREATE OR REPLACE TABLE `qwiklabs-gcp-01-9ed82c2a1f67.fraud_detection.fraud_training_data` AS
SELECT
    Applicant_ID,
    Age,
    Income,
    Number_of_Dependents,
    Amount_Requested,
    Application_Frequency_Last_Year,
    Fraudulent,

    -- One-hot: Employment_Status
    CASE WHEN Employment_Status = 'Employed' THEN 1 ELSE 0 END AS Employment_Status_Employed,
    CASE WHEN Employment_Status = 'Self-Employed' THEN 1 ELSE 0 END AS Employment_Status_Self_Employed,
    CASE WHEN Employment_Status = 'Unemployed' THEN 1 ELSE 0 END AS Employment_Status_Unemployed,

    -- One-hot: Device_Type
    CASE WHEN Device_Type = 'Desktop' THEN 1 ELSE 0 END AS Device_Type_Desktop,
    CASE WHEN Device_Type = 'Mobile' THEN 1 ELSE 0 END AS Device_Type_Mobile,
    CASE WHEN Device_Type = 'Tablet' THEN 1 ELSE 0 END AS Device_Type_Tablet,

    -- Age bins (one-hot)
    CASE WHEN Age BETWEEN 18 AND 24 THEN 1 ELSE 0 END AS Age_18_24,
    CASE WHEN Age BETWEEN 25 AND 34 THEN 1 ELSE 0 END AS Age_25_34,

In [8]:
run(feature_sql)
print("Created table:", TRAIN_TABLE)


Created table: qwiklabs-gcp-01-9ed82c2a1f67.fraud_detection.fraud_training_data


In [9]:
client.query(f"SELECT * FROM `{TRAIN_TABLE}` LIMIT 10").to_dataframe()

,Applicant_ID,Age,Income,Number_of_Dependents,Amount_Requested,Application_Frequency_Last_Year,Fraudulent,Employment_Status_Employed,Employment_Status_Self_Employed,Employment_Status_Unemployed,...,Age_35_44,Age_45_54,Age_55_64,Age_65_plus,Income_to_Amount_Requested,Time_Since_Previous_Assistance_Raw,Time_Since_Previous_Assistance,Has_Previous_Assistance,Previous_Assistance_Received,Supporting_Doc_Verified
0,1193,61,33854,5,620,1,0,1,0,0,...,0,0,1,0,54.603226,<NA>,0,0,0,1
1,4172,27,56567,1,1853,1,0,1,0,0,...,0,0,0,0,30.527253,<NA>,0,0,0,1
2,4571,24,0,2,2667,1,0,0,1,0,...,0,0,0,0,0.000000,<NA>,0,0,0,0
3,4769,30,0,2,4168,1,0,1,0,0,...,0,0,0,0,0.000000,<NA>,0,0,0,1
4,5025,29,31682,0,6833,1,0,0,1,0,...,0,0,0,0,4.636616,<NA>,0,0,0,1
5,5077,61,70557,4,3414,1,0,0,1,0,...,0,0,1,0,20.666960,<NA>,0,0,0,0
6,5292,32,18695,2,7440,1,0,0,0,1,...,0,0,0,0,2.512769,<NA>,0,0,0,1
7,8117,54,19419,1,2334,1,0,0,1,0,...,0,1,0,0,8.320051,<NA>,0,0,0,0
8,9101,37,71308,4,3050,1,0,1,0,0,...,1,0,0,0,23.379672,<NA>,0,0,0,1
9,11394,53,0,3,8512,1,0,0,1,0,...,0,1,0,0,0.000000,<NA>,0,0,0,0


In [10]:
# Sanity checks
import pandas as pd
pd.set_option("display.max_columns", None)

schema = client.get_table(TRAIN_TABLE)
print("Rows:", schema.num_rows)
print("Columns:", len(schema.schema))
for f in schema.schema:
    print(f" - {f.name}: {f.field_type}")


Rows: 50000
Columns: 25
 - Applicant_ID: INTEGER
 - Age: INTEGER
 - Income: INTEGER
 - Number_of_Dependents: INTEGER
 - Amount_Requested: INTEGER
 - Application_Frequency_Last_Year: INTEGER
 - Fraudulent: INTEGER
 - Employment_Status_Employed: INTEGER
 - Employment_Status_Self_Employed: INTEGER
 - Employment_Status_Unemployed: INTEGER
 - Device_Type_Desktop: INTEGER
 - Device_Type_Mobile: INTEGER
 - Device_Type_Tablet: INTEGER
 - Age_18_24: INTEGER
 - Age_25_34: INTEGER
 - Age_35_44: INTEGER
 - Age_45_54: INTEGER
 - Age_55_64: INTEGER
 - Age_65_plus: INTEGER
 - Income_to_Amount_Requested: FLOAT
 - Time_Since_Previous_Assistance_Raw: INTEGER
 - Time_Since_Previous_Assistance: INTEGER
 - Has_Previous_Assistance: INTEGER
 - Previous_Assistance_Received: INTEGER
 - Supporting_Doc_Verified: INTEGER


In [11]:
# Confirm the ratio and the time-since field behave as expected
client.query(f"""
SELECT
  Applicant_ID, Income, Amount_Requested, Income_to_Amount_Requested,
  Has_Previous_Assistance, Time_Since_Previous_Assistance,
  Previous_Assistance_Received, Supporting_Doc_Verified
FROM `{TRAIN_TABLE}`
ORDER BY Applicant_ID
LIMIT 10
""").to_dataframe()


,Applicant_ID,Income,Amount_Requested,Income_to_Amount_Requested,Has_Previous_Assistance,Time_Since_Previous_Assistance,Previous_Assistance_Received,Supporting_Doc_Verified
0,201,37829,686,55.144315,0,0,0,1
1,202,0,5650,0.000000,0,0,0,1
2,203,0,1849,0.000000,0,0,0,1
3,204,0,8468,0.000000,1,216,1,1
4,205,40349,8932,4.517353,1,359,1,1
5,206,0,7499,0.000000,0,0,0,1
6,207,0,3756,0.000000,1,460,1,0
7,208,0,3877,0.000000,0,0,0,1
8,209,0,2219,0.000000,1,208,1,0
9,210,0,9834,0.000000,0,0,0,1
